### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="houses",
    dataset_year="1990",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://lib.stat.cmu.edu/datasets/",
    download_description="""
We download the houses.zip from the lib.stat.cmu.edu repository and preprocess cadata.txt (strip the 27-line header text, remove leading whitespace, add a column header row).

mkdir -p local-data-warehouse/houses/ && wget -q -P /tmp https://lib.stat.cmu.edu/datasets/houses.zip && unzip -o /tmp/houses.zip -d /tmp && { echo "MedianHouseValue  MedianIncome  HousingMedianAge  TotalRooms  TotalBedrooms  Population  Households  Latitude  Longitude"; sed -n '28,$p' /tmp/cadata.txt | sed 's/^  //'; } > local-data-warehouse/houses/cadata_manual.txt && rm /tmp/houses.zip /tmp/cadata.txt
""",
    # References
    academic_reference_bibtex=r"""@article{pace1997sparse,
  title={Sparse spatial autoregressions},
  author={Pace, R Kelley and Barry, Ronald},
  journal={Statistics \\& Probability Letters},
  volume={33},
  number={3},
  pages={291--297},
  year={1997},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="pace1997sparse",
    license="Public",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- The data contains rows whose values were capped artificially at a price of 500001. This creates a censored target, punishing model that learn to extrapolate from the data. We remove rows with this censored value in the target variable to obtain a more realistic task.
- We kept the latitude and longitude features as they are.
- We log scaled the target variable (with base e) as intended by the original task.
- Anomaly: As always, we randomly shuffle the data before uploading. If one does not randomly shuffle the data, there would be a distribution shift from the longitude and latitude based on the original order of data samples.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LnMedianHouseValue",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/cadata_manual.txt", index_col=False, sep="  ")

# Fix lat/longitude whitespace error from original data
df[["Latitude", "Longitude"]] = df["Latitude"].str.split(" ", expand=True)
df[["Latitude", "Longitude"]] = df[["Latitude", "Longitude"]].astype(float)

df = df[df["MedianHouseValue"] < 500001].reset_index(drop=True)


target_feature = "LnMedianHouseValue"

# Transform to log space as defined by original task
df[target_feature] = np.log(df["MedianHouseValue"])
df = df.drop(columns=["MedianHouseValue"])

# Spots a distribution shift in the dataset based on the order of the samples, removed by shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()